# CEFR 3-Band Classification via a 0-100 Score - 2 Methods (baseline + score reshaping: 2 variations)

**Business pipeline (fixed):** each method produces a **0-100 score**, then **2 cut-points**
on that score split it into the **3 bands**.

```
features -> model -> class probabilities -> 0-100 score -> 2 cut-points -> 3 bands
```

**Bands:**

| Band | CEFR levels | Code |
|---|---|---|
| 0 | A1, A2 | `A1-A2` |
| 1 | B1 | `B1` |
| 2 | B2, C1, C2 | `B2-C1-C2` |

**Baseline to beat:** 77% (senior's regression models). **Target:** >=82%.

**This is the simple baseline version:**
- **No hyperparameter tuning** - fixed, sensible defaults.
- **No cross-validation / CV accuracy** shown.
- Reports plain **TRAIN**, **TEST**, and **FULL (train+test)** accuracy.

The two methods are the classic tree ensembles, both on the same Frank-Hall ordinal skeleton:

| # | Method | Family | One-line intuition |
|---|---|---|---|
| 1 | **Ordinal Random Forest** (Frank-Hall) | Bagged trees | Many independent trees, averaged - kills variance |
| 2 | **Ordinal Boosting** (LightGBM + Frank-Hall) | Boosted trees | Trees added in sequence, each fixing the last - kills bias |

## 0. Imports

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    cohen_kappa_score, confusion_matrix, classification_report,
)

import lightgbm as lgb

print("lightgbm loaded")

## 1. Load your data  <-- FILL THIS IN

Assign your dataframe to `df`. Nothing else in the notebook needs changing here.

In [ ]:
# TODO: initialise your dataframe here
df = None

# e.g. df = pd.read_csv("your_file.csv")
# e.g. df = pd.read_excel("your_file.xlsx")

## 2. Columns  <-- FILL THIS IN

Put your chosen features (one per feature group, 8-11 total) in `FEATURE_COLS`.
Everything else is metadata: `ciid`, `location`, `split`, and the label.

In [ ]:
# TODO: one feature per feature group (8-11 total)
FEATURE_COLS = []

# OPTIONAL: map each feature to its feature group / assessment section, so the
# importance tables in every method can be reported per section.
# Leave empty to treat every feature as its own section.
FEATURE_GROUPS = {}     # e.g. {"lex_div": "Vocabulary", "mlu": "Grammar"}

# ---- metadata columns (edit names if yours differ) ----
ID_COL       = "ciid"
LOCATION_COL = "location"
SPLIT_COL    = "split"
LABEL_COL    = "cefr"       # TODO: your CEFR label column

# values inside SPLIT_COL that define the sets; anything else is dropped (bad flags)
TRAIN_VALUE = "train"
TEST_VALUE  = "test"

META_COLS = [ID_COL, LOCATION_COL, SPLIT_COL, LABEL_COL]

## 3. Configuration

In [ ]:
RANDOM_STATE = 42

# Band definition: A1,A2 -> 0 | B1 -> 1 | B2,C1,C2 -> 2
BAND_MAP = {
    "A1": 0, "A2": 0,
    "B1": 1,
    "B2": 2, "C1": 2, "C2": 2,
}
BAND_NAMES = ["A1-A2", "B1", "B2-C1-C2"]
N_BANDS = 3

# Score anchors: score = sum_k P(band_k) * anchor_k  -> lands in [0, 100].
# Equally spaced anchors -> the two cut-points, once tuned, give the same accuracy
# regardless of the absolute values; only the printed number changes.
BAND_ANCHORS = np.array([0.0, 50.0, 100.0])

OPTIMIZE_METRIC = "accuracy"     # or "balanced_accuracy" if bands are imbalanced

BASELINE_ACC = 0.77              # senior's regression baseline
TARGET_ACC   = 0.82              # goal

PERM_REPEATS = 20                # permutation-importance repeats

## 4. Build train / test  (shared by both methods)

Rows whose `split` is neither `train` nor `test` (the "bad flags") are dropped.

In [ ]:
assert df is not None, "Assign your dataframe to `df` in section 1."
assert len(FEATURE_COLS) > 0, "Fill FEATURE_COLS in section 2."

missing = [c for c in FEATURE_COLS + META_COLS if c not in df.columns]
assert not missing, f"Columns not found in df: {missing}"


def to_band(s):
    """Map CEFR labels to band 0/1/2. Accepts 6-level strings or already-coded 0/1/2."""
    s = pd.Series(s)
    if s.dtype.kind in "iuf":
        vals = set(pd.unique(s.dropna()))
        if vals <= {0, 1, 2}:
            return s.astype(int).to_numpy()
    key = s.astype(str).str.strip().str.upper().str.replace(" ", "", regex=False)
    mapped = key.map(BAND_MAP)
    bad = key[mapped.isna()].unique()
    assert len(bad) == 0, f"Unmapped label values: {bad}"
    return mapped.astype(int).to_numpy()


split_norm = df[SPLIT_COL].astype(str).str.strip().str.lower()
train_mask = split_norm == TRAIN_VALUE
test_mask  = split_norm == TEST_VALUE
dropped    = (~train_mask & ~test_mask).sum()

train_df = df.loc[train_mask].copy()
test_df  = df.loc[test_mask].copy()

X_train = train_df[FEATURE_COLS].astype(float)
X_test  = test_df[FEATURE_COLS].astype(float)
y_train = to_band(train_df[LABEL_COL])
y_test  = to_band(test_df[LABEL_COL])

print(f"features           : {len(FEATURE_COLS)}  -> {FEATURE_COLS}")
print(f"train / test rows  : {len(X_train)} / {len(X_test)}")
print(f"dropped (bad flags): {dropped}")
print(f"other split values : {sorted(set(split_norm) - {TRAIN_VALUE, TEST_VALUE})}")
print()
print("train band distribution:")
print(pd.Series(y_train).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))
print()
print("test band distribution:")
print(pd.Series(y_test).value_counts().sort_index()
        .rename(lambda i: f"{i} = {BAND_NAMES[i]}"))

## 5. Core utilities

- `FrankHallOrdinal` - ordinal decomposition: `K-1` binary models for `P(y > k)`, differenced.
- `proba_to_score` - probabilities -> the **0-100 score**.
- `fit_cutpoints` - grid-search the **2 cut-points** (on train scores).
- `evaluate` - runs the full pipeline for one method and reports TRAIN / TEST / FULL accuracy.
- `report_importance` - native + permutation importance, rolled up to sections.

In [ ]:
from cefr_common import FrankHallOrdinal   # shared class so saved models load in cefr_inference.ipynb


def make_pipe(estimator, scale=True):
    steps = [("impute", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scale", StandardScaler()))
    steps.append(("model", estimator))
    return Pipeline(steps)


def proba_to_score(proba, anchors=BAND_ANCHORS):
    return np.asarray(proba) @ np.asarray(anchors, dtype=float)


def apply_cutpoints(scores, t1, t2):
    scores = np.asarray(scores)
    return np.where(scores <= t1, 0, np.where(scores <= t2, 1, 2))


def _scorer(name):
    return accuracy_score if name == "accuracy" else balanced_accuracy_score


def fit_cutpoints(scores, y, metric=OPTIMIZE_METRIC, n_grid=120):
    """Exhaustive grid search for the 2 thresholds maximising `metric`."""
    scores = np.asarray(scores)
    sc = _scorer(metric)
    cand = np.unique(np.percentile(scores, np.linspace(0, 100, n_grid)))
    best_val, best_cuts = -1.0, (33.3, 66.7)
    for i in range(len(cand) - 1):
        for j in range(i + 1, len(cand)):
            v = sc(y, apply_cutpoints(scores, cand[i], cand[j]))
            if v > best_val:
                best_val, best_cuts = v, (float(cand[i]), float(cand[j]))
    return best_cuts, best_val


RESULTS, FITTED, IMPORTANCE = [], {}, {}


def _metrics(y, pred):
    return dict(
        acc=accuracy_score(y, pred),
        bal_acc=balanced_accuracy_score(y, pred),
        macro_f1=f1_score(y, pred, average="macro"),
        qwk=cohen_kappa_score(y, pred, weights="quadratic"),
        mae=float(np.mean(np.abs(np.asarray(y) - np.asarray(pred)))),
    )


def evaluate(name, model, anchors=BAND_ANCHORS, verbose=True):
    """Fit on train -> probabilities -> 0-100 score -> 2 cut-points -> TRAIN / TEST / FULL.

    Baseline simplicity: the cut-points are fitted on the TRAIN scores. The test set
    is never used to pick them, so `test_acc` is still an honest held-out number.
    (The more rigorous option fits cut-points on out-of-fold scores - see the doc.)
    """
    model.fit(X_train, y_train)
    p_tr, p_te = model.predict_proba(X_train), model.predict_proba(X_test)
    s_tr, s_te = proba_to_score(p_tr, anchors), proba_to_score(p_te, anchors)

    (t1, t2), _ = fit_cutpoints(s_tr, y_train)          # cut-points from TRAIN scores
    pred_tr = apply_cutpoints(s_tr, t1, t2)
    pred_te = apply_cutpoints(s_te, t1, t2)

    y_full = np.concatenate([y_train, y_test])
    pred_full = np.concatenate([pred_tr, pred_te])

    m_te = _metrics(y_test, pred_te)
    train_acc = accuracy_score(y_train, pred_tr)
    full_acc = accuracy_score(y_full, pred_full)
    argmax_acc = accuracy_score(y_test, np.argmax(p_te, axis=1))

    row = dict(method=name,
               train_acc=train_acc, test_acc=m_te["acc"], full_acc=full_acc,
               test_bal_acc=m_te["bal_acc"], macro_f1=m_te["macro_f1"],
               qwk=m_te["qwk"], mae=m_te["mae"],
               argmax_acc=argmax_acc, lift_vs_argmax=m_te["acc"] - argmax_acc,
               cut1=t1, cut2=t2)
    RESULTS.append(row)
    FITTED[name] = dict(model=model, score_test=s_te, pred_test=pred_te,
                        score_train=s_tr, pred_train=pred_tr,
                        proba_test=p_te, proba_train=p_tr, cuts=(t1, t2))

    if verbose:
        flag = "PASS >=82%" if m_te["acc"] >= TARGET_ACC else (
               "beats 77%" if m_te["acc"] >= BASELINE_ACC else "below 77%")
        print(f"  TRAIN accuracy : {train_acc:.3f}")
        print(f"  TEST  accuracy : {m_te['acc']:.3f}   <-- the honest number  [{flag}]")
        print(f"  FULL  accuracy : {full_acc:.3f}   (train+test; inflated, do not quote)")
        print(f"  cut-points {t1:.1f} / {t2:.1f} | bal {m_te['bal_acc']:.3f} | "
              f"QWK {m_te['qwk']:.3f} | argmax {argmax_acc:.3f} (lift {m_te['acc']-argmax_acc:+.3f})")
    return row


print("utilities ready")

### Importance helpers

Two views are produced for **each** method:

- **Native importance** - what the tree ensemble itself exposes (impurity reduction for the
  forest, split gain for LightGBM), reported **per cumulative question** - which sections
  matter at the low end (`P(y>0)`) versus the high end (`P(y>1)`).
- **Permutation importance** - shuffle one feature, re-run the *entire* pipeline
  (model -> 0-100 score -> cut-points) and measure the drop in band accuracy. Measured on
  **TRAIN, TEST and FULL (train+test)** so you can compare all three.

Both are rolled up to assessment sections via `FEATURE_GROUPS`.

In [ ]:
def banded_predict(model, X, t1, t2, anchors=BAND_ANCHORS):
    """Full pipeline: probabilities -> 0-100 score -> cut-points -> band."""
    return apply_cutpoints(proba_to_score(model.predict_proba(X), anchors), t1, t2)


def permutation_importance_banded(model, X, y, t1, t2, n_repeats=None, seed=RANDOM_STATE):
    """Permutation importance measured on the FINAL band prediction."""
    n_repeats = n_repeats or PERM_REPEATS
    rng = np.random.default_rng(seed)
    base = accuracy_score(y, banded_predict(model, X, t1, t2))
    recs = []
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].to_numpy())
            drops.append(base - accuracy_score(y, banded_predict(model, Xp, t1, t2)))
        recs.append((col, float(np.mean(drops)), float(np.std(drops))))
    return (pd.DataFrame(recs, columns=["feature", "importance", "std"])
              .sort_values("importance", ascending=False).reset_index(drop=True)), base


def frankhall_native_importance(fh, how="gain"):
    """Native importance per cumulative question of a Frank-Hall model."""
    prof = {}
    for k, (kind, m) in enumerate(fh.models_):
        if kind != "model":
            continue
        est = m.steps[-1][1] if hasattr(m, "steps") else m
        vals = None
        if hasattr(est, "booster_"):
            try:
                vals = est.booster_.feature_importance(importance_type=how)
            except Exception:
                vals = None
        if vals is None:
            vals = getattr(est, "feature_importances_", None)
        if vals is not None:
            v = np.asarray(vals, dtype=float)
            prof[f"P(y>{k})"] = v / v.sum() if v.sum() else v
    if not prof:
        return None
    out = pd.DataFrame(prof, index=FEATURE_COLS)
    out["mean"] = out.mean(axis=1)
    return out.sort_values("mean", ascending=False)


def report_importance(name, native=None, native_note=""):
    """Native importance + permutation importance measured on TRAIN, TEST and FULL."""
    fit = FITTED[name]
    t1, t2 = fit["cuts"]
    X_full = pd.concat([X_train, X_test])
    y_full = np.concatenate([y_train, y_test])

    p_tr, b_tr = permutation_importance_banded(fit["model"], X_train, y_train, t1, t2)
    p_te, b_te = permutation_importance_banded(fit["model"], X_test,  y_test,  t1, t2)
    p_fu, b_fu = permutation_importance_banded(fit["model"], X_full,  y_full,  t1, t2)

    imp = (p_tr.rename(columns={"importance": "imp_train"})[["feature", "imp_train"]]
           .merge(p_te.rename(columns={"importance": "imp_test"})[["feature", "imp_test"]], on="feature")
           .merge(p_fu.rename(columns={"importance": "imp_full"})[["feature", "imp_full"]], on="feature"))
    imp["section"] = imp["feature"].map(lambda f: FEATURE_GROUPS.get(f, f))
    imp = imp.sort_values("imp_test", ascending=False).reset_index(drop=True)
    IMPORTANCE[name] = dict(native=native, importance=imp)

    if native is not None:
        print(f"NATIVE importance{(' - ' + native_note) if native_note else ''}")
        nat = native.copy()
        nat["section"] = [FEATURE_GROUPS.get(f, f) for f in nat.index]
        display(nat.round(4))

    print("\nPERMUTATION importance = drop in band accuracy when a feature is shuffled")
    print(f"  measured on 3 sets | base accuracy: "
          f"train {b_tr:.3f} | test {b_te:.3f} | full {b_fu:.3f}")
    display(imp[["feature", "section", "imp_train", "imp_test", "imp_full"]].round(4))

    weak = imp[imp["imp_test"] <= 0]["feature"].tolist()
    if weak:
        print(f"no help on test: {weak}")
    return imp


print("importance helpers ready")

---

# Method 1 - Ordinal Random Forest (Frank-Hall)

*Current best performer.*

### How it works
The three bands are **ordered**, so instead of one 3-class problem we ask **two cumulative
yes/no questions** (Frank-Hall decomposition):

| Model | Question | Positive label |
|---|---|---|
| RF #1 | is this learner **above band 0**? (`y > 0`) | B1, B2-C1-C2 |
| RF #2 | is this learner **above band 1**? (`y > 1`) | B2-C1-C2 |

Each question gets its own Random Forest (hundreds of decision trees, each on a bootstrap
resample of the rows and each split restricted to a random subset of features). The output
probability is the fraction of trees voting "yes".

Writing `c0 = P(y>0)` and `c1 = P(y>1)`, band probabilities come back by differencing:
`P(band0) = 1 - c0`, `P(band1) = c0 - c1`, `P(band2) = c1`.

### Why it suits this problem
Bagging attacks **variance**, the dominant failure mode at n~220, and it is unbothered by the
feature groups being correlated proxies of the same construct.

*Fixed defaults used below - no tuning at this baseline stage.*

In [ ]:
print("Method 1 - Ordinal Random Forest (Frank-Hall)")

rf = RandomForestClassifier(
    n_estimators=800, max_features="sqrt", min_samples_leaf=3,
    class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1,
)
M1 = "1. Ordinal RF (Frank-Hall)"
evaluate(M1, make_pipe(FrankHallOrdinal(rf), scale=False))

In [ ]:
fh_rf = FITTED[M1]["model"].steps[-1][1]
report_importance(M1,
                  native=frankhall_native_importance(fh_rf),
                  native_note="impurity reduction, per cumulative question")

---

# Method 2 - Ordinal Boosting (LightGBM + Frank-Hall)

### How it works
Same Frank-Hall skeleton, but each cumulative question is answered by a **gradient-boosted
ensemble**. Boosting builds trees **sequentially** - each new tree is fitted to the errors the
current ensemble still makes, so it steadily self-corrects. Trees are kept shallow
(`max_depth` 2-4) and combined at a low learning rate.

### Bagging vs boosting - the contrast to study
- **Random Forest (bagging):** trees in **parallel**, averaged. Reduces **variance**. More
  trees never overfits.
- **Boosting:** trees in **sequence**, each depending on the last. Reduces **bias**. More
  trees *can* overfit - hence the conservative fixed settings below.

They fail in different ways, so agreement between them is a good sign.

*Fixed defaults used below - no tuning at this baseline stage.*

In [ ]:
print("Method 2 - Ordinal Boosting (LightGBM + Frank-Hall)")

gbm = lgb.LGBMClassifier(
    n_estimators=400, learning_rate=0.03, max_depth=3, num_leaves=7,
    min_child_samples=15, subsample=0.8, subsample_freq=1,
    colsample_bytree=0.8, reg_lambda=1.0,
    random_state=RANDOM_STATE, verbose=-1,
)
M2 = "2. Ordinal Boosting (LGBM + Frank-Hall)"
evaluate(M2, make_pipe(FrankHallOrdinal(gbm), scale=False))

In [ ]:
fh_gbm = FITTED[M2]["model"].steps[-1][1]
report_importance(M2,
                  native=frankhall_native_importance(fh_gbm, how="gain"),
                  native_note="LightGBM gain, per cumulative question")

---

# Results

## Leaderboard

| Column | Meaning | Honest? |
|---|---|---|
| `train_acc` | in-sample: fitted on train, predicted on train | **No** - optimistic |
| `test_acc`  | held-out test set | **Yes** - the number to report |
| `full_acc`  | train + test combined | **No** - inflated by the in-sample train part |

- **Report `test_acc`** against the 77% baseline. `train_acc` and `full_acc` are diagnostics.
- A big `train_acc - test_acc` gap just means the model fits its training data tightly
  (normal for tree ensembles); what matters is that `test_acc` holds up.

In [ ]:
lb = (pd.DataFrame(RESULTS).sort_values("test_acc", ascending=False)
        .reset_index(drop=True))
lb["vs_baseline"] = lb["test_acc"] - BASELINE_ACC
lb["status"] = np.where(lb["test_acc"] >= TARGET_ACC, "PASS >=82%",
                 np.where(lb["test_acc"] >= BASELINE_ACC, "beats 77%", "below 77%"))
lb["train_test_gap"] = lb["train_acc"] - lb["test_acc"]

pd.set_option("display.width", 200, "display.max_columns", 50)
display(lb[["method", "train_acc", "test_acc", "full_acc", "train_test_gap",
            "test_bal_acc", "macro_f1", "qwk", "mae",
            "argmax_acc", "lift_vs_argmax", "cut1", "cut2",
            "vs_baseline", "status"]].round(4))

print(f"\nbaseline {BASELINE_ACC:.0%} | target {TARGET_ACC:.0%}")
print(f"best (by TEST accuracy): {lb.iloc[0]['method']} -> {lb.iloc[0]['test_acc']:.3f}")

## Does the 0-100 score beat plain `argmax`?

Positive `lift_vs_argmax` means the score + cut-points classify better than reading the
model's raw top prediction - i.e. the business 0-100 step is also earning its keep.

In [ ]:
crux = lb[["method", "test_acc", "argmax_acc", "lift_vs_argmax"]].sort_values(
    "lift_vs_argmax", ascending=False)
display(crux.round(4))

## Feature importance on TRAIN, TEST and FULL - one percentage each

For each set (train / test / whole dataset) the permutation importance is averaged over the
two methods and turned into a **share of the total** that sums to ~100%. So every feature gets
three comparable percentages side by side (`pct_train`, `pct_test`, `pct_full`), plus a
roll-up to assessment **sections**. Raw importances (`imp_*`) are shown alongside.

In [ ]:
def _pct(series):
    pos = series.clip(lower=0)
    return (100 * pos / pos.sum()).round(1) if pos.sum() > 0 else pos * 0.0


comp = pd.DataFrame(index=FEATURE_COLS)
for col, tag in [("imp_train", "train"), ("imp_test", "test"), ("imp_full", "full")]:
    means = pd.concat([d["importance"].set_index("feature")[col]
                       for d in IMPORTANCE.values()], axis=1).mean(axis=1)
    comp[f"imp_{tag}"] = means
    comp[f"pct_{tag}"] = _pct(means)
comp["section"] = [FEATURE_GROUPS.get(f, f) for f in comp.index]
comp = comp.sort_values("pct_test", ascending=False)

print("Feature importance as a single % on each set (averaged over both methods)")
print("each pct_* column sums to ~100%\n")
display(comp[["section", "pct_train", "pct_test", "pct_full",
              "imp_train", "imp_test", "imp_full"]].round(3))

never = comp[comp["imp_test"] <= 0].index.tolist()
print(f"\ndrop candidates (0% on test): {never}")

if FEATURE_GROUPS:
    print("\nSECTION-level percentage (each column sums to ~100%):")
    sect = (comp.groupby("section")[["pct_train", "pct_test", "pct_full"]].sum()
                .sort_values("pct_test", ascending=False).round(1))
    display(sect)

## Confusion matrices - train, test and full - for the best method

In [ ]:
best_name = lb.iloc[0]["method"]
best = FITTED[best_name]
t1, t2 = best["cuts"]
print(f"=== {best_name} ===  cut-points: {t1:.1f} / {t2:.1f}")

y_full = np.concatenate([y_train, y_test])
pred_full = np.concatenate([best["pred_train"], best["pred_test"]])

for title, yt, yp in [("TRAIN", y_train, best["pred_train"]),
                      ("TEST", y_test, best["pred_test"]),
                      ("FULL (train+test)", y_full, pred_full)]:
    print(f"\n----- {title} confusion matrix  (accuracy {accuracy_score(yt, yp):.3f}) -----")
    display(pd.DataFrame(confusion_matrix(yt, yp, labels=[0, 1, 2]),
                         index=[f"true {b}" for b in BAND_NAMES],
                         columns=[f"pred {b}" for b in BAND_NAMES]))

print("\n----- TEST classification report -----")
print(classification_report(y_test, best["pred_test"], target_names=BAND_NAMES))

## Soft-voting ensemble of the two

Averaging the probability vectors of the forest (low variance) and the booster (low bias)
often nudges accuracy up - but on a small test set, verify it actually helps.

In [ ]:
members = list(FITTED)
P_tr = np.mean([FITTED[m]["proba_train"] for m in members], axis=0)
P_te = np.mean([FITTED[m]["proba_test"] for m in members], axis=0)

s_tr, s_te = proba_to_score(P_tr), proba_to_score(P_te)
(t1e, t2e), _ = fit_cutpoints(s_tr, y_train)

pred_tr, pred_te = apply_cutpoints(s_tr, t1e, t2e), apply_cutpoints(s_te, t1e, t2e)
m = _metrics(y_test, pred_te)

print(f"ENSEMBLE of {len(members)}: {members}")
print(f"  TRAIN {accuracy_score(y_train, pred_tr):.3f} | TEST {m['acc']:.3f}")
print(f"  bal {m['bal_acc']:.3f} | macroF1 {m['macro_f1']:.3f} | QWK {m['qwk']:.3f}")
print(f"  cut-points ({t1e:.1f}, {t2e:.1f})")
print(f"\nbest single method: {lb.iloc[0]['test_acc']:.3f} "
      f"-> ensemble: {m['acc']:.3f}  ({m['acc'] - lb.iloc[0]['test_acc']:+.3f})")

## Final 0-100 scores for the test set

The business deliverable: a per-row 0-100 score plus the band it falls into.

In [ ]:
out = pd.DataFrame({
    ID_COL:       test_df[ID_COL].values,
    LOCATION_COL: test_df[LOCATION_COL].values,
    "score_0_100": np.round(FITTED[best_name]["score_test"], 2),
    "pred_band":   [BAND_NAMES[i] for i in FITTED[best_name]["pred_test"]],
    "true_band":   [BAND_NAMES[i] for i in y_test],
})
out["correct"] = out["pred_band"] == out["true_band"]
display(out.head(20))

print(f"\nmethod: {best_name}")
print(f"cut-points: {FITTED[best_name]['cuts'][0]:.1f} / {FITTED[best_name]['cuts'][1]:.1f}")
print(f"test accuracy: {out['correct'].mean():.3f}")

# out.to_csv("test_scores.csv", index=False)

---

# Appendix - the 0-100 conversion and the split points

## A.1 From the two scores to a 0-100 number

Each method outputs **two** numbers, one per Frank-Hall question:

- `c0 = P(y > 0)` - probability the learner is **above A1-A2** (B1 or higher)
- `c1 = P(y > 1)` - probability the learner is **above B1** (B2-C1-C2)

**Step 1 - two scores to three band probabilities** (differencing):
```
P(A1-A2)    = 1  - c0
P(B1)       = c0 - c1
P(B2-C1-C2) =      c1        (sums to 1)
```

**Step 2 - three probabilities to one score** (weighted average over anchors 0/50/100):
```
score = P(A1-A2)*0 + P(B1)*50 + P(B2-C1-C2)*100
```

**Step 3 - the algebra collapses:**
```
score = (1-c0)*0 + (c0-c1)*50 + c1*100 = 50*(c0 + c1)
      = 50 * ( P(y>0) + P(y>1) )
```

**Why it is always in [0, 100].** `c0, c1` are probabilities in [0,1], so `c0+c1` is in [0,2]
and `*50` lands in [0,100]. (Equivalently, a weighted average can never leave the range of
the anchors it averages.)

**Intuition.** Each of the two proficiency bars a learner clears is worth 50 points, weighted
by the model's confidence they cleared it. Worked example, `c0=0.90, c1=0.30`:
`50*(0.90+0.30) = 60`, matching `P=(0.10,0.60,0.30) -> 0*.1+50*.6+100*.3 = 60`.

**Why an expected value, not `argmax`.** `argmax` discards confidence: two learners both
labelled "B1" (scores 30 vs 70) look identical to it. The 0-100 score separates them, which
is exactly what the business wants - and it lets the cut-points recover accuracy `argmax`
leaves behind (the `lift_vs_argmax` column).

## A.2 How the two split points are decided

**Why not fixed 33.3 / 66.7?** That assumes scores are spread evenly and the model is
perfectly calibrated. In reality scores clump and are shifted, so a fixed 66.7 can slice
through a dense cluster and misclassify a whole group. The cut-points must sit where the model
**actually** separates the bands - an empirical question.

**The search.**
1. Take 120 **percentiles** of the score distribution as candidate positions (so candidates
   land where the learners actually are).
2. Try **every** ordered pair `(t1 < t2)` - ~7,000 combinations.
3. Score each by accuracy of `band = 0 if s<=t1, 1 if s<=t2, else 2`.
4. Keep the best pair. Exhaustive, so it is the global optimum; `33/66` is just one of the
   pairs it tries, so the result is always at least as good.

**Baseline choice: cut-points fit on TRAIN scores.** For simplicity here they are chosen on
the training scores. The **test set is never used to pick them**, so `test_acc` remains an
honest held-out number - at worst the cut-points are slightly sub-optimal (a conservative
baseline, never an inflated one). The more rigorous upgrade fits them on out-of-fold
predictions; that is explained in full in `docs/CEFR_2_Methods_Explained.md` (sections 6, and
the OOF walk-through). Switch to it once you move past baseline.

## A.3 Is the optimum a plateau or a spike?
A single best pair means little if accuracy collapses when a threshold moves by a point.
The next cell measures how wide the near-optimal region is: a **broad plateau** means stable
thresholds that transfer; a **narrow spike** means they are fitted to noise.

In [ ]:
sel = best_name
s_best = FITTED[sel]["score_train"]
t1b, t2b = FITTED[sel]["cuts"]
cand = np.unique(np.percentile(s_best, np.linspace(0, 100, 120)))

rows = []
for a in range(len(cand) - 1):
    for b in range(a + 1, len(cand)):
        rows.append((cand[a], cand[b],
                     accuracy_score(y_train, apply_cutpoints(s_best, cand[a], cand[b]))))
grid = (pd.DataFrame(rows, columns=["t1", "t2", "train_acc"])
          .sort_values("train_acc", ascending=False).reset_index(drop=True))

print(f"{sel}: searched {len(grid):,} threshold pairs\n")
display(grid.head(10).round(3))

best_v = grid["train_acc"].iloc[0]
w1 = (grid["train_acc"] >= best_v - 0.01).sum()
print(f"best train accuracy  : {best_v:.3f}")
print(f"pairs within 1 point : {w1:,}  ({w1/len(grid):.1%} of the grid)")
print(f"chosen cut-points    : {t1b:.2f} / {t2b:.2f}")

top = grid[grid["train_acc"] >= best_v - 0.01]
print(f"\nplateau spans t1 in [{top['t1'].min():.1f}, {top['t1'].max():.1f}], "
      f"t2 in [{top['t2'].min():.1f}, {top['t2'].max():.1f}]")
print("wide plateau => stable thresholds; narrow spike => fitted to noise.")

---

# Reshape the score distribution - two variations (both plotted)

The raw `score = 50*(P(y>0)+P(y>1))` piles up at 0 and 100 (tree ensembles are overconfident for
clear cases). Two fixes are computed and graphed so you can compare:

**A - Global bell (rank -> Beta(5,5)).** Each learner's percentile is mapped onto a single
symmetric bell centred at 50; most land in **35-65**, nobody at 0/100. The score becomes a pure
cohort percentile.

**B - Per-band ranges.** Each band is mapped into its **own** range - **band 0 -> ~2-40**,
**B1 -> 40-60** (centred 50), **band 2 -> ~60-98** - and spread smoothly inside. Bands stay
visibly separated on the score axis; no piling.

Both are monotonic within their scheme, so **bands and accuracy are unchanged**. The final table
carries `raw_score`, `bell_score` (A) and `perband_score` (B) for every learner - pick whichever
the business prefers. Tune with `GLOBAL_BETA` (A) and `BAND_RANGES` / `BAND_BETA` (B);
`RESHAPE = False` reverts.

In [ ]:
from scipy.stats import beta as _beta
from sklearn.preprocessing import QuantileTransformer

RESHAPE = True
_EPS = 1e-3

# ---------- variation A: one global bell, centred ~50 ----------
GLOBAL_BETA = 5.0        # higher = tighter around 50

def fit_global(s_train):
    return QuantileTransformer(output_distribution="uniform",
                               n_quantiles=min(len(s_train), 1000),
                               subsample=1000000000, random_state=RANDOM_STATE
                               ).fit(np.asarray(s_train, float).reshape(-1, 1))

def apply_global(qt, s):
    p = np.clip(qt.transform(np.asarray(s, float).reshape(-1, 1)).ravel(), _EPS, 1 - _EPS)
    return 100.0 * _beta.ppf(p, GLOBAL_BETA, GLOBAL_BETA)

# ---------- variation B: per-band target ranges ----------
BAND_RANGES = {0: (2.0, 40.0), 1: (40.0, 60.0), 2: (60.0, 98.0)}
BAND_BETA = 2.0          # within-band shape: 1.0 flat, higher = more centred

def fit_perband(s_train, bands_train):
    s_train = np.asarray(s_train, float); bands_train = np.asarray(bands_train); t = {}
    for b in range(N_BANDS):
        s = s_train[bands_train == b]
        t[b] = (QuantileTransformer(output_distribution="uniform",
                                    n_quantiles=min(len(s), 1000), subsample=1000000000,
                                    random_state=RANDOM_STATE).fit(s.reshape(-1, 1))
                ) if len(s) >= 2 else None
    return t

def apply_perband(t, s, bands):
    s = np.asarray(s, float); bands = np.asarray(bands); out = np.empty(len(s), float)
    for b in range(N_BANDS):
        m = bands == b
        if not m.any():
            continue
        lo, hi = BAND_RANGES[b]; qt = t.get(b)
        if qt is None:
            out[m] = (lo + hi) / 2.0
        else:
            p = np.clip(qt.transform(s[m].reshape(-1, 1)).ravel(), _EPS, 1 - _EPS)
            out[m] = lo + (hi - lo) * _beta.ppf(p, BAND_BETA, BAND_BETA)
    return out

for nm in FITTED:
    f = FITTED[nm]
    if RESHAPE:
        gq = fit_global(f["score_train"])
        f["reshaper_global"] = gq
        f["score_train_bell"] = apply_global(gq, f["score_train"])
        f["score_test_bell"]  = apply_global(gq, f["score_test"])
        f["cuts_bell"] = tuple(apply_global(gq, np.array(f["cuts"], float)))
        pb = fit_perband(f["score_train"], f["pred_train"])
        f["reshaper_perband"] = pb
        f["score_train_band"] = apply_perband(pb, f["score_train"], f["pred_train"])
        f["score_test_band"]  = apply_perband(pb, f["score_test"], f["pred_test"])
        f["cuts_band"] = (BAND_RANGES[0][1], BAND_RANGES[1][1])   # 40 / 60
    else:
        for suf in ("bell", "band"):
            f[f"score_train_{suf}"] = np.asarray(f["score_train"], float)
            f[f"score_test_{suf}"]  = np.asarray(f["score_test"], float)
            f[f"cuts_{suf}"] = f["cuts"]
    for suf in ("bell", "band"):     # both remaps must preserve every band
        mm = int((apply_cutpoints(f[f"score_test_{suf}"], *f[f"cuts_{suf}"]) != f["pred_test"]).sum())
        if mm:
            print(f"note: {mm} rows changed band ({suf}) for {nm}")

best_name = pd.DataFrame(RESULTS).sort_values("test_acc", ascending=False).iloc[0]["method"]
f = FITTED[best_name]
raw  = np.concatenate([f["score_train"], f["score_test"]])
bell = np.concatenate([f["score_train_bell"], f["score_test_bell"]])
bandS = np.concatenate([f["score_train_band"], f["score_test_band"]])
bnd  = np.concatenate([f["pred_train"], f["pred_test"]])

print(best_name)
print("  raw  percentiles :", np.round(np.percentile(raw, [0, 25, 50, 75, 100]), 1))
print(f"  A global-bell    : {np.round(np.percentile(bell,[0,25,50,75,100]),1)}"
      f"  (in 35-65: {np.mean((bell>=35)&(bell<=65))*100:.0f}%, min {bell.min():.0f} max {bell.max():.0f})")
print("  B per-band ranges:")
for b in range(N_BANDS):
    v = bandS[bnd == b]
    if len(v):
        print(f"     {BAND_NAMES[b]:<9} {v.min():.0f}-{v.max():.0f}  (median {np.median(v):.0f})")

try:
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(1, 3, figsize=(13, 3.2))
    ax[0].hist(raw,   bins=20, color="#c0553b"); ax[0].set_title("raw (U-shaped)")
    ax[1].hist(bell,  bins=20, color="#3b6ea5"); ax[1].set_title("A: global bell (~50)")
    ax[2].hist(bandS, bins=20, color="#2e8b6f"); ax[2].set_title("B: per-band ranges")
    for a in ax:
        a.set_xlim(0, 100); a.set_xlabel("0-100 score"); a.set_ylabel("learners")
    plt.tight_layout(); plt.show()
except Exception as e:
    print("(plot skipped:", e, ")")

## Variation comparison - split points + train/test/full accuracy

Every reshaping is monotonic, so all variations produce the **same bands** -> train/test/full
accuracy are **identical**. This table confirms that (reshaping is lossless for classification)
and shows the **split points** each variation uses, so you can judge which presentation to ship.
Choose by the score distribution you prefer; accuracy does not depend on it.

In [ ]:
_rows = []
for nm in FITTED:
    f = FITTED[nm]
    m = "rf" if "RF" in nm else "gbm"
    for vname, (s_tr, s_te, cuts) in {
        "raw":         (f["score_train"],      f["score_test"],      f["cuts"]),
        "bell (A)":    (f["score_train_bell"], f["score_test_bell"], f["cuts_bell"]),
        "perband (B)": (f["score_train_band"], f["score_test_band"], f["cuts_band"]),
    }.items():
        p_tr = apply_cutpoints(s_tr, *cuts)
        p_te = apply_cutpoints(s_te, *cuts)
        p_fu = np.concatenate([p_tr, p_te])
        _rows.append(dict(model=m, variation=vname,
                          t1=round(float(cuts[0]), 1), t2=round(float(cuts[1]), 1),
                          train_acc=round(accuracy_score(y_train, p_tr), 3),
                          test_acc=round(accuracy_score(y_test, p_te), 3),
                          full_acc=round(accuracy_score(np.concatenate([y_train, y_test]), p_fu), 3)))

cmp = pd.DataFrame(_rows)
display(cmp)
print("Accuracy is identical across variations (monotonic reshape -> same bands).")
print("So pick by the DISTRIBUTION you want; the split points show where each variation cuts.")

---

# Full-dataset predictions - both models, with sub-model scores

Every learner (train **and** test), with **both approaches** (`rf` and `gbm`) side by side.
Each approach exposes its two individual **Frank-Hall sub-model scores** plus the final score
and band:

- `{k}_m1` = **model 1** = `P(y>0)` = probability the learner is **above A1-A2**
- `{k}_m2` = **model 2** = `P(y>1)` = probability the learner is **above B1**
- `{k}_score` = the **0-100** score = `50 * (m1 + m2)`
- `{k}_band` = the predicted band

next to `ciid`, `split` and `region`.

In [ ]:
def _short(nm):
    if "RF" in nm: return "rf"
    if ("Boost" in nm) or ("LGBM" in nm) or ("GBM" in nm): return "gbm"
    return nm.split(".")[0].strip().lower()


def frankhall_cumulative(pipe, X):
    """The two individual sub-model outputs per row: c0 = P(y>0), c1 = P(y>1).

    Monotonic-adjusted (c0 >= c1), exactly as they feed the 0-100 score.
    """
    fh = pipe.steps[-1][1]
    Xt = X
    for _, step in pipe.steps[:-1]:
        Xt = step.transform(Xt)
    cols = [np.full(len(X), m) if kind == "const" else m.predict_proba(Xt)[:, 1]
            for kind, m in fh.models_]
    return np.minimum.accumulate(np.column_stack(cols), axis=1)   # (n, 2): c0, c1


n_tr = len(X_train)
full_pred = pd.DataFrame({
    ID_COL:   np.concatenate([train_df[ID_COL].values,       test_df[ID_COL].values]),
    "region": np.concatenate([train_df[LOCATION_COL].values, test_df[LOCATION_COL].values]),
    "split":  np.array(["train"] * n_tr + ["test"] * len(X_test)),
    "true_band": [BAND_NAMES[i] for i in np.concatenate([y_train, y_test])],
})

# per approach: the two sub-model scores (m1, m2), the final 0-100 score, and the band
for nm in FITTED:
    k = _short(nm)
    f = FITTED[nm]
    cum = np.vstack([frankhall_cumulative(f["model"], X_train),
                     frankhall_cumulative(f["model"], X_test)])
    full_pred[f"{k}_m1"]    = np.round(cum[:, 0], 4)   # model 1: P(y>0), above A1-A2
    full_pred[f"{k}_m2"]    = np.round(cum[:, 1], 4)   # model 2: P(y>1), above B1
    full_pred[f"{k}_raw_score"]     = np.round(np.concatenate([f["score_train"], f["score_test"]]), 2)
    full_pred[f"{k}_bell_score"]    = np.round(np.concatenate([f["score_train_bell"], f["score_test_bell"]]), 2)
    full_pred[f"{k}_perband_score"] = np.round(np.concatenate([f["score_train_band"], f["score_test_band"]]), 2)
    full_pred[f"{k}_band"]  = [BAND_NAMES[i] for i in
                               np.concatenate([f["pred_train"], f["pred_test"]])]

print(f"full dataset: {len(full_pred)} rows ({n_tr} train + {len(X_test)} test) x {len(FITTED)} models")
print("m1=P(y>0) | m2=P(y>1) | raw_score=50*(m1+m2) | bell_score=global bell (A) | perband_score=per-band (B)")
with pd.option_context("display.max_rows", 400, "display.max_columns", 50):
    display(full_pred)

# full_pred.to_csv("full_predictions.csv", index=False)   # uncomment to export every row

---

# Save all models for inference

Bundle both fitted pipelines (impute -> Frank-Hall -> two tree ensembles), their cut-points,
and both reshapers (global bell + per-band) into a single `cefr_models.joblib`. The inference
notebook `cefr_inference.ipynb` loads this and scores new learners with one call.

Needs `cefr_common.py` next to the notebook (it holds the `FrankHallOrdinal` class so the saved
models load cleanly here and in the inference notebook).

In [ ]:
import joblib

_short = lambda nm: "rf" if "RF" in nm else "gbm"

bundle = {
    "feature_cols": list(FEATURE_COLS),
    "band_names": BAND_NAMES,
    "band_anchors": BAND_ANCHORS,
    "reshape_cfg": {"global_beta": GLOBAL_BETA, "band_ranges": BAND_RANGES, "band_beta": BAND_BETA},
    "models": {},
}
for nm in FITTED:
    f = FITTED[nm]; k = _short(nm)
    bundle["models"][k] = {
        "name": nm,
        "pipe": f["model"],
        "cuts": tuple(float(x) for x in f["cuts"]),
        "cuts_bell": tuple(float(x) for x in f["cuts_bell"]),
        "cuts_band": tuple(float(x) for x in f["cuts_band"]),
        "reshaper_global": f["reshaper_global"],
        "reshaper_perband": f["reshaper_perband"],
    }

MODEL_PATH = "cefr_models.joblib"
joblib.dump(bundle, MODEL_PATH)
print("saved", MODEL_PATH)
print("  models   :", list(bundle["models"]))
print("  features :", len(bundle["feature_cols"]))
print("  reshape  :", bundle["reshape_cfg"])